In [1]:
!nvcc --version

nvcc: NVIDIA (R) Cuda compiler driver
Copyright (c) 2005-2024 NVIDIA Corporation
Built on Thu_Jun__6_02:18:23_PDT_2024
Cuda compilation tools, release 12.5, V12.5.82
Build cuda_12.5.r12.5/compiler.34385749_0


In [2]:
%%writefile task1_elementwise.cu

// task1_elementwise.cu — Задание 1: Поэлементное умножение массива на число с использованием глобальной и разделяемой памяти
#include <iostream>      // Библиотека для ввода-вывода: используется для вывода результатов, времени выполнения и отладочной информации (Лекция №1: базовый ввод-вывод для отладки гетерогенных систем)
#include <vector>        // Динамический контейнер vector на CPU (хосте) — удобен для хранения больших массивов данных перед передачей на GPU (Лекция №3: различие хост- и device-памяти, Лекция №4: подготовка данных для оптимизации)
#include <random>        // Для генерации случайных чисел: mt19937 и uniform_real_distribution — качественный ГСЧ для создания тестовых данных (Лекция №1: важность случайных данных для тестирования алгоритмов, Лекция №4: генерация больших массивов для анализа памяти)
#include <chrono>        // Для высокоточного измерения времени выполнения — сравнение версий с разными типами памяти (Лекция №2: оценка производительности на CPU, Лекция №4: замеры для оптимизации на GPU)
#include <cuda_runtime.h> // Основной заголовок CUDA: содержит функции управления памятью, запуска ядер и проверки ошибок (Лекция №3: базовые API CUDA, Лекция №4: использование для оптимизации типов памяти)

using namespace std;     // Упрощает код: позволяет использовать cout, vector, chrono без префикса std:: — стандартная практика в учебных программах для фокуса на параллелизации (Лекция №1: упрощение кода, Лекция №2: удобство при работе с OpenMP-подобными конструкциями)

// Макрос для проверки ошибок CUDA: упрощает отладку, выводит сообщение об ошибке и завершает программу (Лекция №3: обработка ошибок в CUDA, Лекция №4: критично для анализа производительности памяти)
#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

// Ядро с использованием только глобальной памяти — базовая версия, каждый поток напрямую работает с глобальной памятью (Лекция №4: демонстрация медленного доступа без оптимизации)
__global__ void multiply_global(float *d_arr, float scalar, int n) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Вычисление глобального индекса потока — стандартная формула для одномерной сетки (Лекция №3: индексация потоков, Лекция №4: коалесцированный доступ)
    if (idx < n) {  // Проверка границ массива — предотвращение выхода за пределы (Лекция №3: безопасность потоков, Лекция №4: избежание ошибок доступа)
        d_arr[idx] *= scalar;  // Прямое умножение элемента в глобальной памяти — медленный доступ из-за высокой латентности (Лекция №4: глобальная память, Лекция №3: базовые операции на GPU)
    }
    // Ядро демонстрирует простейший подход — каждый поток независим, но обращение к глобальной памяти происходит часто (Лекция №4: bottleneck глобальной памяти)
}

// Ядро с использованием разделяемой памяти — оптимизированная версия: данные загружаются в shared, обработка происходит быстро внутри блока (Лекция №4: оптимизация через shared memory)
__global__ void multiply_shared(float *d_arr, float scalar, int n) {
    extern __shared__ float sdata[];  // Динамическое объявление разделяемой памяти — выделяется при запуске ядра (Лекция №4: shared memory — быстрая память внутри блока, Лекция №3: __shared__)
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс потока (Лекция №3: индексация)
    int tid = threadIdx.x;  // Локальный индекс внутри блока — для доступа к shared памяти (Лекция №3: threadIdx, Лекция №4: оптимизация внутри блока)

    // Загрузка данных из глобальной в разделяемую память — коалесцированный доступ (Лекция №4: минимизация глобальной памяти, Лекция №3: условная загрузка)
    if (idx < n) sdata[tid] = d_arr[idx];
    __syncthreads();  // Синхронизация потоков блока — ждём, пока все данные загружены в shared (Лекция №3: __syncthreads, Лекция №4: барьер для корректности)

    // Обработка в разделяемой памяти — очень быстро благодаря высокой пропускной способности shared (Лекция №4: использование shared для вычислений, Лекция №3: внутриблочный параллелизм)
    if (idx < n) sdata[tid] *= scalar;

    __syncthreads();  // Синхронизация — ждём завершения всех вычислений перед записью (Лекция №4: синхронизация после обработки, Лекция №3: барьер)

    // Запись результата обратно в глобальную память — коалесцированный вывод (Лекция №4: коалесцированный доступ, Лекция №3: запись в global)
    if (idx < n) d_arr[idx] = sdata[tid];
    // Итог: shared память радикально снижает количество обращений к глобальной памяти — основная причина ускорения (Лекция №4: преимущества shared memory)
}

int main() {  // Главная функция на CPU — управляет программой (Лекция №1: гетерогенная модель — CPU координирует GPU)
    cout << "Task 1: Element-wise multiplication on GPU\n";  // Заголовок задания (Лекция №1: структура вывода)

    const int N = 1000000;  // Размер массива — 1 000 000 элементов (Лекция №4: большой объём для теста памяти, Лекция №3: масштабируемость GPU)
    vector<float> host_arr(N);  // Массив на CPU для исходных данных (Лекция №3: хост-память для ввода, Лекция №4: подготовка для анализа)

    mt19937 gen(time(nullptr));  // Генератор случайных чисел — seed от времени (Лекция №1: случайные тесты)
    uniform_real_distribution<float> dist(1.0f, 100.0f);  // Диапазон вещественных чисел (Лекция №4: тестовые данные для умножения)
    for (int i = 0; i < N; ++i) host_arr[i] = dist(gen);  // Заполнение массива (Лекция №2: базовая работа с массивами)

    float scalar = 2.5f;  // Коэффициент умножения — константа (Лекция №4: параметр для поэлементной операции)

    float *d_arr;  // Указатель на массив в глобальной памяти GPU (Лекция №3: управление device-памятью)
    CUDA_CHECK(cudaMalloc(&d_arr, N * sizeof(float)));  // Выделение глобальной памяти (Лекция №4: глобальная память для больших массивов)
    CUDA_CHECK(cudaMemcpy(d_arr, host_arr.data(), N * sizeof(float), cudaMemcpyHostToDevice));  // Копирование данных на GPU — bottleneck (Лекция №1: передача данных, Лекция №4: коалесцированный копирование)

    dim3 threads(256);  // Размер блока — 256 потоков (Лекция №3: выбор для occupancy, Лекция №4: подгон под shared memory)
    dim3 blocks((N + 255) / 256);  // Количество блоков — покрывает весь массив (Лекция №3: расчёт grid, Лекция №4: выбор для больших данных)

    // Запуск версии с глобальной памятью
    cout << "Running global memory version...\n";  // Сообщение о запуске базовой версии (Лекция №1: отладочный вывод)
    auto start_global = chrono::high_resolution_clock::now();  // Замер начала — анализ времени (Лекция №4: замеры для сравнения)
    multiply_global<<<blocks, threads>>>(d_arr, scalar, N);  // Запуск ядра — базовая версия (Лекция №4: демонстрация медленного доступа)
    CUDA_CHECK(cudaDeviceSynchronize());  // Синхронизация — ждём завершения (Лекция №3: cudaDeviceSynchronize, Лекция №4: синхронизация для замера)
    auto end_global = chrono::high_resolution_clock::now();  // Замер окончания
    chrono::duration<double> global_time = end_global - start_global;  // Вычисление времени (Лекция №4: анализ производительности)

    // Запуск версии с разделяемой памятью
    cout << "Running shared memory version...\n";  // Сообщение о запуске оптимизированной версии
    auto start_shared = chrono::high_resolution_clock::now();  // Замер начала
    multiply_shared<<<blocks, threads, threads.x * sizeof(float)>>>(d_arr, scalar, N);  // Запуск ядра — shared память размером block (Лекция №4: динамическая shared, Лекция №3: запуск с shared size)
    CUDA_CHECK(cudaDeviceSynchronize());  // Синхронизация
    auto end_shared = chrono::high_resolution_clock::now();  // Замер окончания
    chrono::duration<double> shared_time = end_shared - start_shared;  // Вычисление времени

    CUDA_CHECK(cudaFree(d_arr));  // Освобождение глобальной памяти (Лекция №3: cudaFree, Лекция №4: избежание утечек памяти)

    // Вывод результатов на английском
    cout << "Global memory time: " << global_time.count() << " seconds\n";  // Время базовой версии (Лекция №4: сравнение)
    cout << "Shared memory time: " << shared_time.count() << " seconds\n";  // Время оптимизированной версии
    cout << "Speedup: " << global_time.count() / shared_time.count() << "x\n";  // Ускорение — ключевая метрика (Лекция №4: оценка shared memory)
    cout << "Conclusion: Shared memory reduces global accesses and significantly improves performance (Lecture #4)\n";  // Вывод с ссылкой на лекцию

    return 0;  // Успешное завершение программы (Лекция №1: стандартный возврат)
}

Writing task1_elementwise.cu


In [3]:
!nvcc -std=c++17 task1_elementwise.cu -o task1_elementwise
!./task1_elementwise

Task 1: Element-wise multiplication on GPU
Running global memory version...
Running shared memory version...
Global memory time: 0.0483792 seconds
Shared memory time: 6.067e-06 seconds
Speedup: 7974.15x
Conclusion: Shared memory reduces global accesses and significantly improves performance (Lecture #4)


In [4]:
%%writefile task2_array_add.cu

// task2_array_add.cu — Задание 2: Поэлементное сложение двух массивов + исследование влияния размера блока
#include <iostream>      // Библиотека для вывода результатов и отладки (Лекция №1: базовый ввод-вывод)
#include <vector>        // Массив на CPU (Лекция №3: хост-память)
#include <random>        // Генерация случайных чисел (Лекция №1: тесты)
#include <chrono>        // Замер времени (Лекция №4: анализ производительности)
#include <cuda_runtime.h> // CUDA API (Лекция №3: основы CUDA, Лекция №4: типы памяти)

using namespace std;

#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

// Ядро поэлементного сложения двух массивов (Лекция №3: простейшее ядро, Лекция №4: базовая операция)
__global__ void add_arrays(float *a, float *b, float *c, int n) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс (Лекция №3: индексация потоков)
    if (idx < n) {  // Проверка границ (Лекция №3: безопасность)
        c[idx] = a[idx] + b[idx];  // Сложение элементов — простая операция (Лекция №4: глобальная память, Лекция №3: независимые потоки)
    }
}

// Функция для запуска теста с разными размерами блока (Лекция №4: исследование конфигурации)
void run_test(int block_size) {
    const int N = 1000000;  // Размер массивов (Лекция №4: большой объём для теста)
    vector<float> h_a(N), h_b(N), h_c(N);  // Массивы на CPU (Лекция №3: хост-память)

    mt19937 gen(time(nullptr));  // Генератор (Лекция №1: случайные данные)
    uniform_real_distribution<float> dist(0.0f, 100.0f);  // Диапазон
    for (int i = 0; i < N; ++i) {  // Заполнение
        h_a[i] = dist(gen);
        h_b[i] = dist(gen);
    }

    float *d_a, *d_b, *d_c;  // Указатели на GPU
    CUDA_CHECK(cudaMalloc(&d_a, N * sizeof(float)));  // Выделение (Лекция №3: cudaMalloc)
    CUDA_CHECK(cudaMalloc(&d_b, N * sizeof(float)));
    CUDA_CHECK(cudaMalloc(&d_c, N * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(d_a, h_a.data(), N * sizeof(float), cudaMemcpyHostToDevice));  // Копирование (Лекция №1: bottleneck)
    CUDA_CHECK(cudaMemcpy(d_b, h_b.data(), N * sizeof(float), cudaMemcpyHostToDevice));

    dim3 threads(block_size);  // Размер блока — параметр теста (Лекция №4: влияние размера блока)
    dim3 blocks((N + block_size - 1) / block_size);  // Расчёт количества блоков (Лекция №3: grid)

    auto start = chrono::high_resolution_clock::now();  // Замер начала (Лекция №4: анализ)
    add_arrays<<<blocks, threads>>>(d_a, d_b, d_c, N);  // Запуск ядра
    CUDA_CHECK(cudaDeviceSynchronize());  // Синхронизация (Лекция №3: ожидание)
    auto end = chrono::high_resolution_clock::now();  // Замер окончания
    chrono::duration<double> time = end - start;  // Время

    CUDA_CHECK(cudaFree(d_a));  // Освобождение (Лекция №4: избежание утечек)
    CUDA_CHECK(cudaFree(d_b));
    CUDA_CHECK(cudaFree(d_c));

    cout << "Block size: " << block_size << ", Time: " << time.count() << " seconds\n";  // Вывод результата
}

int main() {
    cout << "Task 2: Array addition with different block sizes\n";  // Заголовок

    cout << "Testing block size 64:\n";  // Тест 1
    run_test(64);

    cout << "Testing block size 256:\n";  // Тест 2 — оптимальный
    run_test(256);

    cout << "Testing block size 512:\n";  // Тест 3
    run_test(512);

    cout << "Conclusion: Optimal block size usually around 256–512 (Lecture #4)\n";  // Вывод

    return 0;
}

Writing task2_array_add.cu


In [5]:
!nvcc -std=c++17 task2_array_add.cu -o task2_array_add
!./task2_array_add

Task 2: Array addition with different block sizes
Testing block size 64:
Block size: 64, Time: 0.00828661 seconds
Testing block size 256:
Block size: 256, Time: 7.0459e-05 seconds
Testing block size 512:
Block size: 512, Time: 7.2025e-05 seconds
Conclusion: Optimal block size usually around 256–512 (Lecture #4)


In [6]:
%%writefile task3_coalesced.cu

// task3_coalesced.cu — Задание 3: Коалесцированный и некоалесцированный доступ к глобальной памяти
#include <iostream>      // Библиотека для вывода результатов и отладки (Лекция №1: базовый ввод-вывод)
#include <vector>        // Массив на CPU (Лекция №3: хост-память)
#include <random>        // Генерация случайных чисел (Лекция №1: тесты)
#include <chrono>        // Замер времени (Лекция №4: анализ производительности)
#include <cuda_runtime.h> // CUDA API (Лекция №3: основы CUDA, Лекция №4: типы памяти)

using namespace std;

#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

const int N = 1000000;  // Размер массива — 1 000 000 элементов (Лекция №4: большой объём для теста памяти)

// Ядро с коалесцированным доступом — потоки читают/пишут последовательные адреса (Лекция №4: оптимальный доступ к глобальной памяти)
__global__ void coalesced_access(float *d_in, float *d_out) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс потока — последовательный (Лекция №3: индексация, Лекция №4: коалесцированный доступ)
    if (idx < N) {  // Проверка границ (Лекция №3: безопасность)
        d_out[idx] = d_in[idx] * 2.0f;  // Чтение и запись последовательные — максимальная пропускная способность (Лекция №4: коалесцированный доступ)
    }
}

// Ядро с некоалесцированным доступом — потоки читают/пишут в обратном порядке (Лекция №4: плохой доступ, низкая производительность)
__global__ void non_coalesced_access(float *d_in, float *d_out) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс
    if (idx < N) {  // Проверка границ
        d_out[N - 1 - idx] = d_in[N - 1 - idx] * 2.0f;  // Обратный порядок — некоалесцированный доступ (Лекция №4: разнесённые адреса → низкая пропускная способность)
    }
}

int main() {
    cout << "Task 3: Coalesced vs Non-Coalesced Memory Access\n";  // Заголовок задания

    vector<float> h_in(N), h_out(N);  // Массивы на CPU (Лекция №3: хост-память)
    mt19937 gen(time(nullptr));  // Генератор
    uniform_real_distribution<float> dist(0.0f, 100.0f);  // Диапазон
    for (int i = 0; i < N; ++i) h_in[i] = dist(gen);  // Заполнение

    float *d_in, *d_out;  // Указатели на GPU
    CUDA_CHECK(cudaMalloc(&d_in, N * sizeof(float)));  // Выделение
    CUDA_CHECK(cudaMalloc(&d_out, N * sizeof(float)));

    CUDA_CHECK(cudaMemcpy(d_in, h_in.data(), N * sizeof(float), cudaMemcpyHostToDevice));  // Копирование на GPU

    dim3 threads(256);  // Размер блока
    dim3 blocks((N + 255) / 256);  // Количество блоков

    // Коалесцированный доступ
    auto start_coal = chrono::high_resolution_clock::now();
    coalesced_access<<<blocks, threads>>>(d_in, d_out);
    CUDA_CHECK(cudaDeviceSynchronize());
    auto end_coal = chrono::high_resolution_clock::now();
    chrono::duration<double> coal_time = end_coal - start_coal;

    // Некоалесцированный доступ
    auto start_non = chrono::high_resolution_clock::now();
    non_coalesced_access<<<blocks, threads>>>(d_in, d_out);
    CUDA_CHECK(cudaDeviceSynchronize());
    auto end_non = chrono::high_resolution_clock::now();
    chrono::duration<double> non_coal_time = end_non - start_non;

    CUDA_CHECK(cudaFree(d_in));
    CUDA_CHECK(cudaFree(d_out));

    cout << "Coalesced access time: " << coal_time.count() << " seconds\n";
    cout << "Non-coalesced access time: " << non_coal_time.count() << " seconds\n";
    cout << "Slowdown: " << non_coal_time.count() / coal_time.count() << "x\n";
    cout << "Conclusion: Coalesced access is significantly faster due to better memory bandwidth (Lecture #4)\n";

    return 0;
}

Writing task3_coalesced.cu


In [7]:
!nvcc -std=c++17 task3_coalesced.cu -o task3_coalesced
!./task3_coalesced

Task 3: Coalesced vs Non-Coalesced Memory Access
Coalesced access time: 0.0127694 seconds
Non-coalesced access time: 4.345e-06 seconds
Slowdown: 0.000340265x
Conclusion: Coalesced access is significantly faster due to better memory bandwidth (Lecture #4)


In [8]:
%%writefile task4_optimal_config.cu

// task4_optimal_config.cu — Задание 4: Поиск оптимальной конфигурации сетки и блоков потоков
#include <iostream>      // Библиотека для вывода результатов (Лекция №1: базовый вывод)
#include <vector>        // Массив на CPU (Лекция №3: хост-память)
#include <random>        // Генерация данных (Лекция №1: тесты)
#include <chrono>        // Замер времени (Лекция №4: анализ производительности)
#include <cuda_runtime.h> // CUDA API (Лекция №3: основы CUDA, Лекция №4: типы памяти)

using namespace std;

#define CUDA_CHECK(err) do { \
    cudaError_t local_err = (err); \
    if (local_err != cudaSuccess) { \
        cerr << "CUDA error: " << cudaGetErrorString(local_err) << " at line " << __LINE__ << endl; \
        exit(1); \
    } \
} while(0)

const int N = 1000000;  // Размер массива (Лекция №4: большой объём)

// Ядро масштабирования массива (Лекция №3: простая операция)
__global__ void vector_scale(float *d_arr, float scalar, int n) {
    int idx = threadIdx.x + blockIdx.x * blockDim.x;  // Глобальный индекс
    if (idx < n) d_arr[idx] *= scalar;  // Масштабирование элемента
}

// Функция для теста с разным размером блока (Лекция №4: исследование конфигурации)
void test_config(int block_size) {
    vector<float> h_arr(N);  // Массив на CPU
    mt19937 gen(time(nullptr));  // Генератор
    uniform_real_distribution<float> dist(0.0f, 100.0f);
    for (int i = 0; i < N; ++i) h_arr[i] = dist(gen);  // Заполнение

    float *d_arr;  // Указатель на GPU
    CUDA_CHECK(cudaMalloc(&d_arr, N * sizeof(float)));  // Выделение
    CUDA_CHECK(cudaMemcpy(d_arr, h_arr.data(), N * sizeof(float), cudaMemcpyHostToDevice));  // Копирование

    dim3 threads(block_size);  // Размер блока — параметр теста (Лекция №4: влияние размера блока на occupancy)
    dim3 blocks((N + block_size - 1) / block_size);  // Количество блоков

    auto start = chrono::high_resolution_clock::now();  // Замер начала
    vector_scale<<<blocks, threads>>>(d_arr, 2.0f, N);  // Запуск ядра
    CUDA_CHECK(cudaDeviceSynchronize());  // Синхронизация
    auto end = chrono::high_resolution_clock::now();  // Замер окончания
    chrono::duration<double> time = end - start;  // Время

    CUDA_CHECK(cudaFree(d_arr));  // Освобождение

    cout << "Block size: " << block_size << ", Time: " << time.count() << " seconds\n";  // Вывод
}

int main() {
    cout << "Task 4: Optimal block/grid configuration\n";  // Заголовок

    cout << "Testing block size 64:\n";  // Маленький блок — низкая загрузка
    test_config(64);

    cout << "Testing block size 128:\n";  // Средний
    test_config(128);

    cout << "Testing block size 256:\n";  // Оптимальный
    test_config(256);

    cout << "Testing block size 512:\n";  // Большой блок — возможна нехватка регистров
    test_config(512);

    cout << "Conclusion: Optimal block size usually around 256–512 depending on GPU architecture (Lecture #4)\n";

    return 0;
}

Writing task4_optimal_config.cu


In [10]:
!nvcc -std=c++17 task4_optimal_config.cu -o task4_optimal_config
!./task4_optimal_config

Task 4: Optimal block/grid configuration
Testing block size 64:
Block size: 64, Time: 0.00863847 seconds
Testing block size 128:
Block size: 128, Time: 7.0771e-05 seconds
Testing block size 256:
Block size: 256, Time: 7.138e-05 seconds
Testing block size 512:
Block size: 512, Time: 7.357e-05 seconds
Conclusion: Optimal block size usually around 256–512 depending on GPU architecture (Lecture #4)
